In [3]:
library(ggplot2)
library(tidyr)
library(dplyr)
library(stringr)

Warning message:
“package ‘dplyr’ was built under R version 4.3.2”

Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


Warning message:
“package ‘stringr’ was built under R version 4.3.2”


# Create BED files of RNA

In [2]:
genes <- read.table('/nfs/lab/elisha/scripts/refdata-gex-GRCh38-2020-A/genes/genes.gtf.gz', sep='\t')
genes <- filter(genes, V3=='gene')
colnames(genes) <- c('seqnames','source','type','start','end',
                     'score','strand','frame','attribute')
genes$gene_id <- str_remove(str_extract(genes$attribute, 'gene_name[^;]*'), 'gene_name\\s')

#genes[duplicated(genes$gene_id),]
genes$gene_id[which(duplicated(genes$gene_id))] <- paste0(genes$gene_id[which(duplicated(genes$gene_id))], '.1')
#genes[duplicated(genes$gene_id),]

dim(genes)
head(genes)

[1] 36601    10

,seqnames,source,type,start,end,score,strand,frame,attribute,gene_id
,<chr>,<chr>,<chr>,<int>,<int>,<chr>,<chr>,<chr>,<chr>,<chr>
1,chr1,HAVANA,gene,29554,31109,.,+,.,gene_id ENSG00000243485; gene_version 5; gene_type lncRNA; gene_name MIR1302-2HG; level 2; hgnc_id HGNC:52482; tag ncRNA_host; havana_gene OTTHUMG00000000959.2;,MIR1302-2HG
2,chr1,HAVANA,gene,34554,36081,.,-,.,gene_id ENSG00000237613; gene_version 2; gene_type lncRNA; gene_name FAM138A; level 2; hgnc_id HGNC:32334; havana_gene OTTHUMG00000000960.1;,FAM138A
3,chr1,HAVANA,gene,65419,71585,.,+,.,gene_id ENSG00000186092; gene_version 6; gene_type protein_coding; gene_name OR4F5; level 2; hgnc_id HGNC:14825; havana_gene OTTHUMG00000001094.4;,OR4F5
4,chr1,HAVANA,gene,89295,133723,.,-,.,gene_id ENSG00000238009; gene_version 6; gene_type lncRNA; gene_name AL627309.1; level 2; tag overlapping_locus; havana_gene OTTHUMG00000001096.2;,AL627309.1
5,chr1,HAVANA,gene,89551,91105,.,-,.,gene_id ENSG00000239945; gene_version 1; gene_type lncRNA; gene_name AL627309.3; level 2; tag overlapping_locus; havana_gene OTTHUMG00000001097.2;,AL627309.3
6,chr1,HAVANA,gene,139790,140339,.,-,.,gene_id ENSG00000239906; gene_version 1; gene_type lncRNA; gene_name AL627309.2; level 2; havana_gene OTTHUMG00000002481.1;,AL627309.2


In [3]:
gene_gtf <- genes %>%
    mutate(tss = ifelse(strand == "+", start, end),
           # Create BED columns
           chrom = seqnames,
           chromStart = tss - 1,  # BED is 0-based
           chromEnd = tss,
           name = gene_id) %>%
    select(`#chr`=chrom, start=chromStart, end=chromEnd, gene_id=name)

dim(gene_gtf)
head(gene_gtf)

[1] 36601     4

,#chr,start,end,gene_id
,<chr>,<dbl>,<int>,<chr>
1,chr1,29553,29554,MIR1302-2HG
2,chr1,36080,36081,FAM138A
3,chr1,65418,65419,OR4F5
4,chr1,133722,133723,AL627309.1
5,chr1,91104,91105,AL627309.3
6,chr1,140338,140339,AL627309.2


In [4]:
make_count_bed <- function(cell.type, int.filename, bed.filename, gene_gtf, keep_X=TRUE) {
    exp.mat <- read.table(int.filename)
    ct_gtf <- filter(gene_gtf, gene_id %in% rownames(exp.mat))
    
    if(nrow(ct_gtf) != nrow(exp.mat)) {
        print(paste0("Error: Gene name mismatch for ", cell.type))
    }
    
    bed.mat <- cbind(ct_gtf, exp.mat)
    
    # Filter out other contigs
    if (keep_X) {
        bed.mat <- filter(bed.mat, `#chr` %in% paste0('chr',c(1:22,'X')))
    } else {
        bed.mat <- filter(bed.mat, `#chr` %in% paste0('chr',c(1:22)))
    }
    
    print(paste0(nrow(bed.mat), " genes for ", cell.type))
    
    bed.mat <- arrange(bed.mat, `#chr`, start)
    write.table(bed.mat, bed.filename, quote=F, row.names=F, col.names=T, sep='\t')
}

In [5]:
unique_cell_types <- c('Endothelial', 'Hepatocytes', 'T', 'HSC', 'Cholangiocyte', 'Myeloid', 'NK', 'B', 'Mast', 'Schwann', 'Bulk')
indir <- '/nfs/lab/projects/nash_nafld_liver/downstream_all/count.matrices/RNA/'
outdir <- '/nfs/lab/projects/nash_nafld_liver/downstream_all/04_eQTLs/counts/'

##### NAME YOUR FILES #####
for (cell.type in unique_cell_types){
    # Not enough stuff for these cell types
    if (cell.type %in% c('Mast','Schwann')) {
        next
    }
    #tmm.filename <- paste(indir, cell.type, '_perdonor.RNA.TMM.tsv', sep = "")
    int.filename <- paste(indir, cell.type, '_perdonor.RNA.INT.tsv', sep = "")
    bed.filename <- paste(outdir, cell.type, '_perdonor.RNA.INT.bed', sep = "")
    make_count_bed(cell.type, int.filename, bed.filename, gene_gtf, keep_X=FALSE)
}

[1] "15465 genes for Endothelial"
[1] "20235 genes for Hepatocytes"
[1] "7028 genes for T"
[1] "13657 genes for HSC"
[1] "11124 genes for Cholangiocyte"
[1] "13734 genes for Myeloid"
[1] "6234 genes for NK"
[1] "6932 genes for B"
[1] "23377 genes for Bulk"


# Create BED files of ATAC

In [2]:
make_count_bed_atac <- function(cell.type, int.filename, bed.filename) {
    exp.mat <- read.table(int.filename)
    
    ct_gtf <- data.frame(str_split(rownames(exp.mat), '-', simplify=T))
    colnames(ct_gtf) <- c('#chr','start','end')
    ct_gtf$gene_id <- rownames(exp.mat)
    
    bed.mat <- cbind(ct_gtf, exp.mat)
    bed.mat <- arrange(bed.mat, `#chr`, start)
    write.table(bed.mat, bed.filename, quote=F, row.names=F, col.names=T, sep='\t')
}

In [3]:
unique_cell_types <- c('Endothelial', 'Hepatocytes', 'T', 'HSC', 'Cholangiocyte', 'Myeloid', 'NK', 'B', 'Mast', 'Schwann', 'Bulk')
indir <- '/nfs/lab/projects/nash_nafld_liver/downstream_all/count.matrices/ATAC/'
outdir <- '/nfs/lab/projects/nash_nafld_liver/downstream_all/05_caQTLs/counts/'

##### NAME YOUR FILES #####
for (cell.type in unique_cell_types){
    # Not enough stuff for these cell types
    if (cell.type %in% c('Mast','Schwann')) {
        next
    }
    #tmm.filename <- paste(indir, cell.type, '_perdonor.ATAC.TMM.tsv', sep = "")
    int.filename <- paste(indir, cell.type, '_perdonor.ATAC.INT.tsv', sep = "")
    bed.filename <- paste(outdir, cell.type, '_perdonor.ATAC.INT.bed', sep = "")
    make_count_bed_atac(cell.type, int.filename, bed.filename)
}

# Create BED files of H3K27ac

In [10]:
make_count_bed_H3K27ac <- function(cell.type, int.filename, bed.filename) {
    exp.mat <- read.table(int.filename)
    
    ct_gtf <- data.frame(str_split(rownames(exp.mat), '[-:]', simplify=T))
    colnames(ct_gtf) <- c('#chr','start','end')
    ct_gtf$gene_id <- rownames(exp.mat)
    
    bed.mat <- cbind(ct_gtf, exp.mat)
    bed.mat <- arrange(bed.mat, `#chr`, start)
    write.table(bed.mat, bed.filename, quote=F, row.names=F, col.names=T, sep='\t')
}

In [11]:
unique_cell_types <- c('Endothelial', 'Hepatocytes', 'T', 'HSC', 'Cholangiocyte', 'Myeloid', 'NK', 'B', 'Mast', 'Schwann', 'Bulk')
indir <- '/nfs/lab/projects/nash_nafld_liver/downstream_all/count.matrices/H3K27ac/'
outdir <- '/nfs/lab/projects/nash_nafld_liver/downstream_all/06_H3K27acQTLs/counts/'

##### NAME YOUR FILES #####
for (cell.type in unique_cell_types){
    # Not enough stuff for these cell types
    if (cell.type %in% c('Mast','Schwann')) {
        next
    }
    #tmm.filename <- paste(indir, cell.type, '_perdonor.H3K27ac.TMM.tsv', sep = "")
    int.filename <- paste(indir, cell.type, '_perdonor.H3K27ac.INT.tsv', sep = "")
    bed.filename <- paste(outdir, cell.type, '_perdonor.H3K27ac.INT.bed', sep = "")
    make_count_bed_H3K27ac(cell.type, int.filename, bed.filename)
}

# Create BED files of H3K27me3

In [12]:
make_count_bed_H3K27me3 <- function(cell.type, int.filename, bed.filename) {
    exp.mat <- read.table(int.filename)
    
    ct_gtf <- data.frame(str_split(rownames(exp.mat), '[-:]', simplify=T))
    colnames(ct_gtf) <- c('#chr','start','end')
    ct_gtf$gene_id <- rownames(exp.mat)
    
    bed.mat <- cbind(ct_gtf, exp.mat)
    bed.mat <- arrange(bed.mat, `#chr`, start)
    write.table(bed.mat, bed.filename, quote=F, row.names=F, col.names=T, sep='\t')
}

In [13]:
unique_cell_types <- c('Endothelial', 'Hepatocytes', 'T', 'HSC', 'Cholangiocyte', 'Myeloid', 'NK', 'B', 'Mast', 'Schwann', 'Bulk')
indir <- '/nfs/lab/projects/nash_nafld_liver/downstream_all/count.matrices/H3K27me3/'
outdir <- '/nfs/lab/projects/nash_nafld_liver/downstream_all/07_H3K27me3QTLs/counts/'

##### NAME YOUR FILES #####
for (cell.type in unique_cell_types){
    # Not enough stuff for these cell types
    if (cell.type %in% c('Mast','Schwann')) {
        next
    }
    #tmm.filename <- paste(indir, cell.type, '_perdonor.H3K27me3.TMM.tsv', sep = "")
    int.filename <- paste(indir, cell.type, '_perdonor.H3K27me3.INT.tsv', sep = "")
    bed.filename <- paste(outdir, cell.type, '_perdonor.H3K27me3.INT.bed', sep = "")
    make_count_bed_H3K27me3(cell.type, int.filename, bed.filename)
}

# Create BED files of H3K27ac - TMM

In [12]:
make_count_bed_H3K27ac <- function(cell.type, int.filename, tmm.filename, bed.filename) {
    int.mat <- read.table(int.filename)
    tmm.mat <- read.table(tmm.filename)
    exp.mat <- tmm.mat[rownames(int.mat),]
    
    ct_gtf <- data.frame(str_split(rownames(exp.mat), '[-:]', simplify=T))
    colnames(ct_gtf) <- c('#chr','start','end')
    ct_gtf$gene_id <- rownames(exp.mat)
    
    bed.mat <- cbind(ct_gtf, exp.mat)
    bed.mat <- arrange(bed.mat, `#chr`, start)
    write.table(bed.mat, bed.filename, quote=F, row.names=F, col.names=T, sep='\t')
}

In [13]:
unique_cell_types <- c('Endothelial', 'Hepatocytes', 'T', 'HSC', 'Cholangiocyte', 'Myeloid', 'NK', 'B', 'Mast', 'Schwann', 'Bulk')
indir <- '/nfs/lab/projects/nash_nafld_liver/downstream_all/count.matrices/H3K27ac/'
outdir <- '/nfs/lab/projects/nash_nafld_liver/downstream_all/06_H3K27acQTLs/counts/'

##### NAME YOUR FILES #####
for (cell.type in unique_cell_types){
    # Not enough stuff for these cell types
    if (cell.type %in% c('Mast','Schwann')) {
        next
    }
    tmm.filename <- paste(indir, cell.type, '_perdonor.H3K27ac.TMM.tsv', sep = "")
    int.filename <- paste(indir, cell.type, '_perdonor.H3K27ac.INT.tsv', sep = "")
    bed.filename <- paste(outdir, cell.type, '_perdonor.H3K27ac.TMM.bed', sep = "")
    make_count_bed_H3K27ac(cell.type, int.filename, tmm.filename, bed.filename)
}

# Create BED files of H3K27me3 - TMM

In [14]:
make_count_bed_H3K27me3 <- function(cell.type, int.filename, tmm.filename, bed.filename) {
    int.mat <- read.table(int.filename)
    tmm.mat <- read.table(tmm.filename)
    exp.mat <- tmm.mat[rownames(int.mat),]
    
    ct_gtf <- data.frame(str_split(rownames(exp.mat), '[-:]', simplify=T))
    colnames(ct_gtf) <- c('#chr','start','end')
    ct_gtf$gene_id <- rownames(exp.mat)
    
    bed.mat <- cbind(ct_gtf, exp.mat)
    bed.mat <- arrange(bed.mat, `#chr`, start)
    write.table(bed.mat, bed.filename, quote=F, row.names=F, col.names=T, sep='\t')
}

In [15]:
unique_cell_types <- c('Endothelial', 'Hepatocytes', 'T', 'HSC', 'Cholangiocyte', 'Myeloid', 'NK', 'B', 'Mast', 'Schwann', 'Bulk')
indir <- '/nfs/lab/projects/nash_nafld_liver/downstream_all/count.matrices/H3K27me3/'
outdir <- '/nfs/lab/projects/nash_nafld_liver/downstream_all/07_H3K27me3QTLs/counts/'

##### NAME YOUR FILES #####
for (cell.type in unique_cell_types){
    # Not enough stuff for these cell types
    if (cell.type %in% c('Mast','Schwann')) {
        next
    }
    tmm.filename <- paste(indir, cell.type, '_perdonor.H3K27me3.TMM.tsv', sep = "")
    int.filename <- paste(indir, cell.type, '_perdonor.H3K27me3.INT.tsv', sep = "")
    bed.filename <- paste(outdir, cell.type, '_perdonor.H3K27me3.TMM.bed', sep = "")
    make_count_bed_H3K27me3(cell.type, int.filename, tmm.filename, bed.filename)
}

# Write out covariates - RNA

In [8]:
construct_cov_mat <- function(pc.filename, geno.pca.filename, meta, n_geno_pcs=10, cov.filename) {
    feat.pca <- read.table(pc.filename)
    colnames(feat.pca) <- paste0('feature.',colnames(feat.pca))
    
    meta.sub <- meta[rownames(feat.pca),]
    
    gene.pca <- read.table(geno.pca.filename)
    rownames(gene.pca) <- NULL
    gene.pca <- tibble::column_to_rownames(gene.pca,var='IID')
    gene.pca <- gene.pca[,paste0('PC', 1:n_geno_pcs)]
    gene.pca <- gene.pca[rownames(feat.pca),]
    colnames(gene.pca) <- paste0('genotype.',colnames(gene.pca))
    
    covar.mat <- cbind(meta.sub, cbind(feat.pca, gene.pca))
    
    covar.mat <- data.frame(t(covar.mat))
    
    write.table(covar.mat, cov.filename, col.names=T, row.names=T, quote=F, sep='\t')
}

In [9]:
meta.filename <- "/nfs/lab/projects/nash_nafld_liver/downstream_all/01_DESEQ_RNA/RNA.meta.tsv"

meta = read.table(meta.filename,sep='\t', header=T)
colnames(meta)
meta <- tibble::column_to_rownames(meta, var='donor_demux')
meta$Gender <- as.integer(factor(meta$Gender))
meta$disease_status <- as.integer(factor(meta$disease_status))
meta$batch <- as.integer(factor(meta$batch, levels=c('firstPR','secondPR','thirdPR','fourthPR')))
meta$condition <- as.integer(factor(meta$condition, levels=c('Control','MASL','MASH','MetALD')))
meta <- dplyr::select(meta, Gender, Age.scaled, BMI.scaled, Fibrosis.stage, condition, batch)

dim(meta)
head(meta)

[1] "donor_demux"          "B"                    "Cholangiocyte"       
 [4] "Endothelial"          "Hepatocytes"          "HSC"                 
 [7] "Mast"                 "Myeloid"              "NK"                  
[10] "Schwann"              "T"                    "batch"               
[13] "disease_status"       "condition"            "Steatosis.grade"     
[16] "Fibrosis.stage"       "Fat.distribution"     "Lobular.inflammation"
[19] "Ballooning"           "Portal.inflammation"  "Hepatocyte.necrosis" 
[22] "MASH.CRN.score..X.8"  "Gender"               "Age"                 
[25] "BMI"                  "BMI.scaled"           "Age.scaled"

[1] 86  6

,Gender,Age.scaled,BMI.scaled,Fibrosis.stage,condition,batch
,<int>,<dbl>,<dbl>,<int>,<int>,<int>
HL150001,2,-0.01553551,-0.6385847,0,2,3
HL150009,2,-0.75778750,-0.3346869,0,2,4
HL150013,2,-0.16398591,0.4052379,2,3,2
HL150015,2,0.87516689,-0.3743258,1,2,4
HL160017,2,-1.20313870,-1.0481859,2,4,4
HL160022,1,-0.16398591,-0.2818352,2,3,2


In [10]:
indir <- '/nfs/lab/projects/nash_nafld_liver/downstream_all/04_eQTLs/covariates/'
outdir <- '/nfs/lab/projects/nash_nafld_liver/downstream_all/04_eQTLs/covariates/'

n_pc <- 15
n_geno_pcs <- 10

geno.pca.filename <- '/nfs/lab/projects/nash_nafld_liver/genotypes/pca/Supplemental_gPCA_data.tsv'

##### NAME YOUR FILES #####
for (cell.type in unique_cell_types){
    # Not enough stuff for these cell types
    if (cell.type %in% c('Mast','Schwann')) {
        next
    }
    pc.filename <- paste(outdir, cell.type, '_perdonor.RNA.INT.', n_pc,'.PCs.tsv', sep = "")
    cov.filename <- paste(outdir, cell.type, '_perdonor.RNA.covariates.tsv', sep = "")

    construct_cov_mat(pc.filename, geno.pca.filename, meta, n_geno_pcs, cov.filename)
}

# Write out covariates - ATAC

In [26]:
construct_cov_mat <- function(pc.filename, geno.pca.filename, meta, n_geno_pcs=10, cov.filename) {
    feat.pca <- read.table(pc.filename)
    colnames(feat.pca) <- paste0('feature.',colnames(feat.pca))
    
    meta.sub <- meta[rownames(feat.pca),]
    
    gene.pca <- read.table(geno.pca.filename)
    rownames(gene.pca) <- NULL
    gene.pca <- tibble::column_to_rownames(gene.pca,var='IID')
    gene.pca <- gene.pca[,paste0('PC', 1:n_geno_pcs)]
    gene.pca <- gene.pca[rownames(feat.pca),]
    colnames(gene.pca) <- paste0('genotype.',colnames(gene.pca))
    
    covar.mat <- cbind(meta.sub, cbind(feat.pca, gene.pca))
    
    covar.mat <- data.frame(t(covar.mat))
    
    write.table(covar.mat, cov.filename, col.names=T, row.names=T, quote=F, sep='\t')
}

In [27]:
meta.filename <- "/nfs/lab/projects/nash_nafld_liver/downstream_all/01_DESEQ_RNA/RNA.meta.tsv"

meta = read.table(meta.filename,sep='\t', header=T)
colnames(meta)
meta <- tibble::column_to_rownames(meta, var='donor_demux')
meta$Gender <- as.integer(factor(meta$Gender))
meta$disease_status <- as.integer(factor(meta$disease_status))
meta$batch <- as.integer(factor(meta$batch, levels=c('firstPR','secondPR','thirdPR','fourthPR')))
meta$condition <- as.integer(factor(meta$condition, levels=c('Control','MASL','MASH','MetALD')))
meta <- dplyr::select(meta, Gender, Age.scaled, BMI.scaled, Fibrosis.stage, condition, batch)

dim(meta)
head(meta)

[1] "donor_demux"          "B"                    "Cholangiocyte"       
 [4] "Endothelial"          "Hepatocytes"          "HSC"                 
 [7] "Mast"                 "Myeloid"              "NK"                  
[10] "Schwann"              "T"                    "batch"               
[13] "disease_status"       "condition"            "Steatosis.grade"     
[16] "Fibrosis.stage"       "Fat.distribution"     "Lobular.inflammation"
[19] "Ballooning"           "Portal.inflammation"  "Hepatocyte.necrosis" 
[22] "MASH.CRN.score..X.8"  "Gender"               "Age"                 
[25] "BMI"                  "BMI.scaled"           "Age.scaled"

[1] 86  6

,Gender,Age.scaled,BMI.scaled,Fibrosis.stage,condition,batch
,<int>,<dbl>,<dbl>,<int>,<int>,<int>
HL150001,2,-0.01553551,-0.6385847,0,2,3
HL150009,2,-0.75778750,-0.3346869,0,2,4
HL150013,2,-0.16398591,0.4052379,2,3,2
HL150015,2,0.87516689,-0.3743258,1,2,4
HL160017,2,-1.20313870,-1.0481859,2,4,4
HL160022,1,-0.16398591,-0.2818352,2,3,2


In [28]:
indir <- '/nfs/lab/projects/nash_nafld_liver/downstream_all/05_caQTLs/covariates/'
outdir <- '/nfs/lab/projects/nash_nafld_liver/downstream_all/05_caQTLs/covariates/'

n_pc <- 15
n_geno_pcs <- 10

geno.pca.filename <- '/nfs/lab/projects/nash_nafld_liver/genotypes/pca/Supplemental_gPCA_data.tsv'

##### NAME YOUR FILES #####
for (cell.type in unique_cell_types){
    # Not enough stuff for these cell types
    if (cell.type %in% c('Mast','Schwann')) {
        next
    }
    pc.filename <- paste(outdir, cell.type, '_perdonor.ATAC.INT.', n_pc,'.PCs.tsv', sep = "")
    cov.filename <- paste(outdir, cell.type, '_perdonor.ATAC.covariates.tsv', sep = "")

    construct_cov_mat(pc.filename, geno.pca.filename, meta, n_geno_pcs, cov.filename)
}

# Write out covariates - H3K27ac

In [14]:
construct_cov_mat <- function(pc.filename, geno.pca.filename, meta, n_geno_pcs=10, cov.filename) {
    feat.pca <- read.table(pc.filename)
    colnames(feat.pca) <- paste0('feature.',colnames(feat.pca))
    
    meta.sub <- meta[rownames(feat.pca),]
    
    gene.pca <- read.table(geno.pca.filename)
    rownames(gene.pca) <- NULL
    gene.pca <- tibble::column_to_rownames(gene.pca,var='IID')
    gene.pca <- gene.pca[,paste0('PC', 1:n_geno_pcs)]
    gene.pca <- gene.pca[rownames(feat.pca),]
    colnames(gene.pca) <- paste0('genotype.',colnames(gene.pca))
    
    covar.mat <- cbind(meta.sub, cbind(feat.pca, gene.pca))
    
    covar.mat <- data.frame(t(covar.mat))
    
    write.table(covar.mat, cov.filename, col.names=T, row.names=T, quote=F, sep='\t')
}

In [15]:
meta.filename <- "/nfs/lab/projects/nash_nafld_liver/downstream_all/01_DESEQ_RNA/RNA.meta.tsv"

meta = read.table(meta.filename,sep='\t', header=T)
colnames(meta)
meta <- tibble::column_to_rownames(meta, var='donor_demux')
meta$Gender <- as.integer(factor(meta$Gender))
meta$disease_status <- as.integer(factor(meta$disease_status))
meta$batch <- as.integer(factor(meta$batch, levels=c('firstPR','secondPR','thirdPR','fourthPR')))
meta$condition <- as.integer(factor(meta$condition, levels=c('Control','MASL','MASH','MetALD')))
meta <- dplyr::select(meta, Gender, Age.scaled, BMI.scaled, Fibrosis.stage, condition, batch)

dim(meta)
head(meta)

[1] "donor_demux"          "B"                    "Cholangiocyte"       
 [4] "Endothelial"          "Hepatocytes"          "HSC"                 
 [7] "Mast"                 "Myeloid"              "NK"                  
[10] "Schwann"              "T"                    "batch"               
[13] "disease_status"       "condition"            "Steatosis.grade"     
[16] "Fibrosis.stage"       "Fat.distribution"     "Lobular.inflammation"
[19] "Ballooning"           "Portal.inflammation"  "Hepatocyte.necrosis" 
[22] "MASH.CRN.score..X.8"  "Gender"               "Age"                 
[25] "BMI"                  "BMI.scaled"           "Age.scaled"

[1] 86  6

,Gender,Age.scaled,BMI.scaled,Fibrosis.stage,condition,batch
,<int>,<dbl>,<dbl>,<int>,<int>,<int>
HL150001,2,-0.01553551,-0.6385847,0,2,3
HL150009,2,-0.75778750,-0.3346869,0,2,4
HL150013,2,-0.16398591,0.4052379,2,3,2
HL150015,2,0.87516689,-0.3743258,1,2,4
HL160017,2,-1.20313870,-1.0481859,2,4,4
HL160022,1,-0.16398591,-0.2818352,2,3,2


In [16]:
indir <- '/nfs/lab/projects/nash_nafld_liver/downstream_all/06_H3K27acQTLs/covariates/'
outdir <- '/nfs/lab/projects/nash_nafld_liver/downstream_all/06_H3K27acQTLs/covariates/'

n_pc <- 15
n_geno_pcs <- 10

geno.pca.filename <- '/nfs/lab/projects/nash_nafld_liver/genotypes/pca/Supplemental_gPCA_data.tsv'

##### NAME YOUR FILES #####
for (cell.type in unique_cell_types){
    # Not enough stuff for these cell types
    if (cell.type %in% c('Mast','Schwann')) {
        next
    }
    pc.filename <- paste(outdir, cell.type, '_perdonor.H3K27ac.INT.', n_pc,'.PCs.tsv', sep = "")
    cov.filename <- paste(outdir, cell.type, '_perdonor.H3K27ac.covariates.tsv', sep = "")

    construct_cov_mat(pc.filename, geno.pca.filename, meta, n_geno_pcs, cov.filename)
}

# Write out covariates - H3K27me3

In [17]:
construct_cov_mat <- function(pc.filename, geno.pca.filename, meta, n_geno_pcs=10, cov.filename) {
    feat.pca <- read.table(pc.filename)
    colnames(feat.pca) <- paste0('feature.',colnames(feat.pca))
    
    meta.sub <- meta[rownames(feat.pca),]
    
    gene.pca <- read.table(geno.pca.filename)
    rownames(gene.pca) <- NULL
    gene.pca <- tibble::column_to_rownames(gene.pca,var='IID')
    gene.pca <- gene.pca[,paste0('PC', 1:n_geno_pcs)]
    gene.pca <- gene.pca[rownames(feat.pca),]
    colnames(gene.pca) <- paste0('genotype.',colnames(gene.pca))
    
    covar.mat <- cbind(meta.sub, cbind(feat.pca, gene.pca))
    
    covar.mat <- data.frame(t(covar.mat))
    
    write.table(covar.mat, cov.filename, col.names=T, row.names=T, quote=F, sep='\t')
}

In [18]:
meta.filename <- "/nfs/lab/projects/nash_nafld_liver/downstream_all/01_DESEQ_RNA/RNA.meta.tsv"

meta = read.table(meta.filename,sep='\t', header=T)
colnames(meta)
meta <- tibble::column_to_rownames(meta, var='donor_demux')
meta$Gender <- as.integer(factor(meta$Gender))
meta$disease_status <- as.integer(factor(meta$disease_status))
meta$batch <- as.integer(factor(meta$batch, levels=c('firstPR','secondPR','thirdPR','fourthPR')))
meta$condition <- as.integer(factor(meta$condition, levels=c('Control','MASL','MASH','MetALD')))
meta <- dplyr::select(meta, Gender, Age.scaled, BMI.scaled, Fibrosis.stage, condition, batch)

dim(meta)
head(meta)

[1] "donor_demux"          "B"                    "Cholangiocyte"       
 [4] "Endothelial"          "Hepatocytes"          "HSC"                 
 [7] "Mast"                 "Myeloid"              "NK"                  
[10] "Schwann"              "T"                    "batch"               
[13] "disease_status"       "condition"            "Steatosis.grade"     
[16] "Fibrosis.stage"       "Fat.distribution"     "Lobular.inflammation"
[19] "Ballooning"           "Portal.inflammation"  "Hepatocyte.necrosis" 
[22] "MASH.CRN.score..X.8"  "Gender"               "Age"                 
[25] "BMI"                  "BMI.scaled"           "Age.scaled"

[1] 86  6

,Gender,Age.scaled,BMI.scaled,Fibrosis.stage,condition,batch
,<int>,<dbl>,<dbl>,<int>,<int>,<int>
HL150001,2,-0.01553551,-0.6385847,0,2,3
HL150009,2,-0.75778750,-0.3346869,0,2,4
HL150013,2,-0.16398591,0.4052379,2,3,2
HL150015,2,0.87516689,-0.3743258,1,2,4
HL160017,2,-1.20313870,-1.0481859,2,4,4
HL160022,1,-0.16398591,-0.2818352,2,3,2


In [19]:
indir <- '/nfs/lab/projects/nash_nafld_liver/downstream_all/07_H3K27me3QTLs/covariates/'
outdir <- '/nfs/lab/projects/nash_nafld_liver/downstream_all/07_H3K27me3QTLs/covariates/'

n_pc <- 15
n_geno_pcs <- 10

geno.pca.filename <- '/nfs/lab/projects/nash_nafld_liver/genotypes/pca/Supplemental_gPCA_data.tsv'

##### NAME YOUR FILES #####
for (cell.type in unique_cell_types){
    # Not enough stuff for these cell types
    if (cell.type %in% c('Mast','Schwann')) {
        next
    }
    pc.filename <- paste(outdir, cell.type, '_perdonor.H3K27me3.INT.', n_pc,'.PCs.tsv', sep = "")
    cov.filename <- paste(outdir, cell.type, '_perdonor.H3K27me3.covariates.tsv', sep = "")

    construct_cov_mat(pc.filename, geno.pca.filename, meta, n_geno_pcs, cov.filename)
}

# Write out covariates - H3K27ac - TMM

In [31]:
construct_cov_mat <- function(pc.filename, geno.pca.filename, meta, n_geno_pcs=10, cov.filename) {
    feat.pca <- read.table(pc.filename)
    colnames(feat.pca) <- paste0('feature.',colnames(feat.pca))
    
    meta.sub <- meta[rownames(feat.pca),]
    
    gene.pca <- read.table(geno.pca.filename)
    rownames(gene.pca) <- NULL
    gene.pca <- tibble::column_to_rownames(gene.pca,var='IID')
    gene.pca <- gene.pca[,paste0('PC', 1:n_geno_pcs)]
    gene.pca <- gene.pca[rownames(feat.pca),]
    colnames(gene.pca) <- paste0('genotype.',colnames(gene.pca))
    
    covar.mat <- cbind(meta.sub, cbind(feat.pca, gene.pca))
    
    covar.mat <- data.frame(t(covar.mat))
    
    write.table(covar.mat, cov.filename, col.names=T, row.names=T, quote=F, sep='\t')
}

In [32]:
meta.filename <- "/nfs/lab/projects/nash_nafld_liver/downstream_all/01_DESEQ_RNA/RNA.meta.tsv"

meta = read.table(meta.filename,sep='\t', header=T)
colnames(meta)
meta <- tibble::column_to_rownames(meta, var='donor_demux')
meta$Gender <- as.integer(factor(meta$Gender))
meta$disease_status <- as.integer(factor(meta$disease_status))
meta$batch <- as.integer(factor(meta$batch, levels=c('firstPR','secondPR','thirdPR','fourthPR')))
meta$condition <- as.integer(factor(meta$condition, levels=c('Control','MASL','MASH','MetALD')))
meta <- dplyr::select(meta, Gender, Age.scaled, BMI.scaled, Fibrosis.stage, condition, batch)

dim(meta)
head(meta)

[1] "donor_demux"          "B"                    "Cholangiocyte"       
 [4] "Endothelial"          "Hepatocytes"          "HSC"                 
 [7] "Mast"                 "Myeloid"              "NK"                  
[10] "Schwann"              "T"                    "batch"               
[13] "disease_status"       "condition"            "Steatosis.grade"     
[16] "Fibrosis.stage"       "Fat.distribution"     "Lobular.inflammation"
[19] "Ballooning"           "Portal.inflammation"  "Hepatocyte.necrosis" 
[22] "MASH.CRN.score..X.8"  "Gender"               "Age"                 
[25] "BMI"                  "BMI.scaled"           "Age.scaled"

[1] 86  6

,Gender,Age.scaled,BMI.scaled,Fibrosis.stage,condition,batch
,<int>,<dbl>,<dbl>,<int>,<int>,<int>
HL150001,2,-0.01553551,-0.6385847,0,2,3
HL150009,2,-0.75778750,-0.3346869,0,2,4
HL150013,2,-0.16398591,0.4052379,2,3,2
HL150015,2,0.87516689,-0.3743258,1,2,4
HL160017,2,-1.20313870,-1.0481859,2,4,4
HL160022,1,-0.16398591,-0.2818352,2,3,2


In [33]:
indir <- '/nfs/lab/projects/nash_nafld_liver/downstream_all/06_H3K27acQTLs/covariates/'
outdir <- '/nfs/lab/projects/nash_nafld_liver/downstream_all/06_H3K27acQTLs/covariates/'

n_pc <- 15
n_geno_pcs <- 10

geno.pca.filename <- '/nfs/lab/projects/nash_nafld_liver/genotypes/pca/Supplemental_gPCA_data.tsv'

##### NAME YOUR FILES #####
for (cell.type in unique_cell_types){
    # Not enough stuff for these cell types
    if (cell.type %in% c('Mast','Schwann')) {
        next
    }
    pc.filename <- paste(outdir, cell.type, '_perdonor.H3K27ac.TMM.', n_pc,'.PCs.tsv', sep = "")
    cov.filename <- paste(outdir, cell.type, '_perdonor.H3K27ac.covariates.TMM.tsv', sep = "")

    construct_cov_mat(pc.filename, geno.pca.filename, meta, n_geno_pcs, cov.filename)
}

# Write out covariates - H3K27me3 - TMM

In [34]:
construct_cov_mat <- function(pc.filename, geno.pca.filename, meta, n_geno_pcs=10, cov.filename) {
    feat.pca <- read.table(pc.filename)
    colnames(feat.pca) <- paste0('feature.',colnames(feat.pca))
    
    meta.sub <- meta[rownames(feat.pca),]
    
    gene.pca <- read.table(geno.pca.filename)
    rownames(gene.pca) <- NULL
    gene.pca <- tibble::column_to_rownames(gene.pca,var='IID')
    gene.pca <- gene.pca[,paste0('PC', 1:n_geno_pcs)]
    gene.pca <- gene.pca[rownames(feat.pca),]
    colnames(gene.pca) <- paste0('genotype.',colnames(gene.pca))
    
    covar.mat <- cbind(meta.sub, cbind(feat.pca, gene.pca))
    
    covar.mat <- data.frame(t(covar.mat))
    
    write.table(covar.mat, cov.filename, col.names=T, row.names=T, quote=F, sep='\t')
}

In [35]:
meta.filename <- "/nfs/lab/projects/nash_nafld_liver/downstream_all/01_DESEQ_RNA/RNA.meta.tsv"

meta = read.table(meta.filename,sep='\t', header=T)
colnames(meta)
meta <- tibble::column_to_rownames(meta, var='donor_demux')
meta$Gender <- as.integer(factor(meta$Gender))
meta$disease_status <- as.integer(factor(meta$disease_status))
meta$batch <- as.integer(factor(meta$batch, levels=c('firstPR','secondPR','thirdPR','fourthPR')))
meta$condition <- as.integer(factor(meta$condition, levels=c('Control','MASL','MASH','MetALD')))
meta <- dplyr::select(meta, Gender, Age.scaled, BMI.scaled, Fibrosis.stage, condition, batch)

dim(meta)
head(meta)

[1] "donor_demux"          "B"                    "Cholangiocyte"       
 [4] "Endothelial"          "Hepatocytes"          "HSC"                 
 [7] "Mast"                 "Myeloid"              "NK"                  
[10] "Schwann"              "T"                    "batch"               
[13] "disease_status"       "condition"            "Steatosis.grade"     
[16] "Fibrosis.stage"       "Fat.distribution"     "Lobular.inflammation"
[19] "Ballooning"           "Portal.inflammation"  "Hepatocyte.necrosis" 
[22] "MASH.CRN.score..X.8"  "Gender"               "Age"                 
[25] "BMI"                  "BMI.scaled"           "Age.scaled"

[1] 86  6

,Gender,Age.scaled,BMI.scaled,Fibrosis.stage,condition,batch
,<int>,<dbl>,<dbl>,<int>,<int>,<int>
HL150001,2,-0.01553551,-0.6385847,0,2,3
HL150009,2,-0.75778750,-0.3346869,0,2,4
HL150013,2,-0.16398591,0.4052379,2,3,2
HL150015,2,0.87516689,-0.3743258,1,2,4
HL160017,2,-1.20313870,-1.0481859,2,4,4
HL160022,1,-0.16398591,-0.2818352,2,3,2


In [36]:
indir <- '/nfs/lab/projects/nash_nafld_liver/downstream_all/07_H3K27me3QTLs/covariates/'
outdir <- '/nfs/lab/projects/nash_nafld_liver/downstream_all/07_H3K27me3QTLs/covariates/'

n_pc <- 15
n_geno_pcs <- 10

geno.pca.filename <- '/nfs/lab/projects/nash_nafld_liver/genotypes/pca/Supplemental_gPCA_data.tsv'

##### NAME YOUR FILES #####
for (cell.type in unique_cell_types){
    # Not enough stuff for these cell types
    if (cell.type %in% c('Mast','Schwann')) {
        next
    }
    pc.filename <- paste(outdir, cell.type, '_perdonor.H3K27me3.TMM.', n_pc,'.PCs.tsv', sep = "")
    cov.filename <- paste(outdir, cell.type, '_perdonor.H3K27me3.covariates.TMM.tsv', sep = "")

    construct_cov_mat(pc.filename, geno.pca.filename, meta, n_geno_pcs, cov.filename)
}

# VCF

# Running tensorQTL

#### RNA

In [ ]:
make.scripts.sh
for file in *_RNA_QTL.sh; do echo sbatch $file; done > run_all_ct.sh 
bash run_all_ct.sh

#### ATAC

In [ ]:
#!/bin/bash

for ct in 'Hepatocytes'; do

    # Create a unique SLURM batch script for each cell type
    batch_script="${ct}_ATAC_QTL_trans.sh"

    # Write the batch script
    cat <<EOT > $batch_script
#!/bin/bash
#SBATCH --job-name=${ct}_ATAC_QTL_trans      # Job name
#SBATCH --output=${ct}_ATAC_QTL_trans%j.out # Output file (%j will be replaced by job ID)
#SBATCH --error=${ct}_ATAC_QTL_trans%j.err  # Error file
#SBATCH --nodes=1                     # Number of nodes
#SBATCH --partition=a100              # Partition
#SBATCH --account=csd854
#SBATCH --qos=condo-gpu
#SBATCH --ntasks=1                    # Number of tasks
#SBATCH --cpus-per-task=1             # CPUs per task (5 cells in parallel, matches -P 5)
#SBATCH --mem=50G                     # Memory per node
#SBATCH --time=48:00:00               # Time limit (48 hours)
#SBATCH --gpus=2                      # Number of GPUs

plink_prefix_path=/tscc/projects/ps-gaultonlab/welison/FNIH.Liver/tensorQTL/05_caQTLs/genotypes/Liver.QTL.r209.maf05.matchednames
expression_bed=/tscc/projects/ps-gaultonlab/welison/FNIH.Liver/tensorQTL/05_caQTLs/counts/${ct}_perdonor.ATAC.INT.sort.1k.bed
prefix=/tscc/projects/ps-gaultonlab/welison/FNIH.Liver/tensorQTL/05_caQTLs/outs/trans/${ct}
covariates_file=/tscc/projects/ps-gaultonlab/welison/FNIH.Liver/tensorQTL/05_caQTLs/covariates/${ct}_perdonor.ATAC.covariates.tsv

echo "Set all variables for ${ct}"

# Run TensorQTL for all SNPs
python3 -m tensorqtl \${plink_prefix_path} \${expression_bed} \${prefix} \
    --covariates \${covariates_file} \
    --mode trans

echo "Done with ${ct}"

EOT

        chmod +x ${ct}_ATAC_QTL_trans.sh

done

In [ ]:
python3 -m tensorqtl ${plink_prefix_path} ${expression_bed} ${prefix} \
    --covariates ${covariates_file} \
    --mode trans

In [ ]:
make.scripts.sh
for file in *_ATAC_QTL.sh; do echo sbatch $file; done > run_all_ct.sh 
for file in *_ATAC_QTL_SuSiE.sh; do echo sbatch $file; done > run_all_ct_SuSiE.sh 
bash run_all_ct.sh
bash run_all_ct_SuSiE.sh

#### H3K27ac

#### H3K27me3 

In [ ]:
bash make.scripts.sh
for file in *_H3K27me3_QTL.sh; do echo sbatch $file; done > run_all_ct.sh 
bash run_all_ct.sh

#### H3K27ac - TMM

#### H3K27me3 

In [ ]:
bash make.scripts.sh
for file in *_H3K27me3_QTL_TMM.sh; do echo sbatch $file; done > run_all_ct_TMM.sh 
bash run_all_ct_TMM.sh
for file in *_H3K27me3_QTL_TMM_SuSiE.sh; do echo sbatch $file; done > run_all_ct_TMM_SuSiE.sh 
bash run_all_ct_TMM_SuSiE.sh